# Build Results Fact
1. Read silver `results` table
1. Read silver `sprints` table
1. Add new column `session_type` with values `RACE` or `SPRINT`
1. UNION `results` and `sprints`
1. Derive additional columns
    - is_win -> Indicates that the driver own the race
    - is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
    - has_points -> Indicates that the driver has scored points
1. Write the transformed data to gold `fact_session_results` table

In [0]:
%run ../00-common/01.enviroment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
target_table = f'{catalog_name}.{gold_schema}.fact_session_results'

#### Step 1 - Read source tables

In [0]:
results_df = spark.read.table(f'{catalog_name}.{silver_schema}.results').withColumn('session_type', F.lit('RACE'))
sprints_df = spark.read.table(f'{catalog_name}.{silver_schema}.sprints').withColumn('session_type', F.lit('SPRINT'))

####Step 4 - UNION `results` and `sprints`

In [0]:
results_sprints_df = (
    results_df
        .unionByName(sprints_df)
)

#### Step 3 - Add dervied columns
1. is_win -> Indicates that the driver own the race
1. is_podium -> Indicates that the driver scored a podium result (1, 2, 3)
1. has_points -> Indicates that the driver has scored points


In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn('is_win', results_sprints_df.finish_position == 1)
        .withColumn('is_podium', results_sprints_df.finish_position.between(1, 3))
        .withColumn('has_points', results_sprints_df.points > 0)
)

In [0]:
display(fact_session_results_df.filter('season == 2022'))

####Step 5 - Write the transformed data to gold `fact_session_results` table

In [0]:
fact_session_results = (
    fact_session_results_df.write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(target_table)
)

In [0]:
%sql
SELECT * FROM formula1.gold.fact_session_results